In [ ]:
from optics_design_workbench.jupyter_utils import *
from matplotlib.pyplot import *
import IPython.display
import threading
import time
import psutil

# start endless simulation in background thread

In [ ]:
t0 = time.time()
def runEndlessSimulation():
  try:
    with FreecadDocument('main.FCStd', showProgress=False) as f:
      f.runSimulation('true')
  # retry once if 'ungently exit' error is raised because this simulation is always exited ungently by this test notebook
  except RuntimeError as e:
    if 'ungently' in str(e):
      time.sleep(30)
      with FreecadDocument('main.FCStd', showProgress=False) as f:
        f.runSimulation('true')
    else:
      raise

thread = threading.Thread(target=runEndlessSimulation, daemon=True)
thread.start()  

In [ ]:
time.sleep(120)

# record memory consumption for 14 hours

Fourteen hours make sure a worker endOfLife is experienced (happens randomly between 10 and 12 hours)

Track memory consumption to make sure slope of memory increase is under control

In [ ]:
memory = []
differentSizeCouter = 0
while True:

  # store memory consumption of children and timestamps
  process = psutil.Process()
  rss = process.memory_info().rss
  memory.append([time.time()])
  for child in process.children(recursive=True):
    memory[-1].append(child.memory_info().rss)

  # make sure at least one child exists
  if len(memory[-1]) <= 1:
    raise ValueError('no child processes found, did the simulation die?')

  # make sure number of children does not change
  if len(memory) > 2 and len(memory[-1]) != len(memory[-2]):
    memory = memory[:-1]
    print(f'warning: {len(memory[-1])} != {len(memory[-2])}, skipping last measurement')
    differentSizeCouter += 1
  else:
    differentSizeCouter = 0
  if differentSizeCouter > 5:
    raise ValueError(f'{len(memory[-1])} != {len(memory[-2])} for multiple times in a row')

  # plot memory vs time
  if len(memory) > 2:
    IPython.display.clear_output(True)
    TMs = array(memory).T
    T, Ms = TMs[0], TMs[1:].T
    figure(figsize=(14,5))
    plot((T-t0)/60**2, Ms/1024**3)
    xlabel('time since start of simulation (hours)')
    ylabel('reserved memory (GiB)')
    show()

    # calculate and check/print memory increase slopes if enough data present
    if len(T) > 10:
      hi = 1
      while True:
        # calc for first hour in data
        _t, _m = T[T<min(T)+60*60], Ms[T<min(T)+60*60]
        GbPerHours = []
        for i,c in enumerate(sorted(_m.T, key=lambda e: e.max())):
          GbPerHour = (c[-1]-c[0])/(_t[-1]-_t[0])*60*60/1024**3
          GbPerHours.append(GbPerHour)
          print(f'hour {hi:2.0f} child {i:2.0f} memory increase: {GbPerHour:6.2f} GiB/hour')
        
        backgroundGain, guiGain = GbPerHours[:-1], GbPerHours[-1]
        if _t[-1]-_t[0] > 50*60 and max(backgroundGain) > 0.1:
          raise ValueError(f'memory increase rate of background workers is out of bounds for at least one child: {backgroundGain=}')
        if _t[-1]-_t[0] > 50*60 and guiGain > 1:
          raise ValueError(f'memory increase rate of GUI process is out of bounds: {guiGain=}')

        # shorten data
        T, Ms = T[T>=min(T)+60*60], Ms[T>=min(T)+60*60]
        hi += 1
        if len(T) < 10:
          break

  # end of loop and limit loop speed
  if time.time()-t0 > 14*60*60:
    print(f'fourteen hours passed, exiting happily')
  time.sleep(30)